# XAI skalibrowanego modelu v2
Próbki losowane warstwowo. Średnie bezwzględne SHAP agregowane według tekstu tokenu, nie jego pozycji. Analiza opisowa nie służy zmianom modelu po obejrzeniu testu.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
# Set PHISHING_RUN_ID / PHISHING_SEED before starting the notebook kernel.
from src.config import RESULTS_DIR, SAVED_MODELS_DIR
import pandas as pd
import numpy as np


In [ ]:
import shap
from src.evaluation.explainer import PhishingExplainer
from src.evaluation.evaluate import load_final_predictions

frame = load_final_predictions()
frame = frame[frame.model == "herbert-base"]
samples = frame.groupby("Is_Phishing", group_keys=False).sample(n=5, random_state=42)
explainer = PhishingExplainer(str(SAVED_MODELS_DIR / "herbert-base"))
contributions = []

try:
    for row in samples.itertuples():
        values = explainer.get_explanation(row.Text)

        for token, impact in zip(values.data[0], values.values[0, :, 1]):
            contributions.append({"token": str(token).strip(), "absolute_impact": abs(float(impact)), "Record_ID": row.Record_ID})

        shap.plots.text(values[0, :, 1])

finally:
    explainer.close()

importance = pd.DataFrame(contributions).groupby("token").absolute_impact.agg(["mean", "count"]).sort_values("mean", ascending=False)
importance.to_csv(RESULTS_DIR / "shap_token_importance.csv")
display(importance.head(20))
